In [ ]:
import os
import random
from collections import defaultdict

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision import datasets
from torchvision.models import resnet50                # ← add this
from PIL import Image
from einops import rearrange, repeat
from einops.layers.torch import Rearrange
from tqdm import tqdm

# ------------------------------
# 1. Config
# ------------------------------
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 1
TARGET_PER_CLASS = 2000
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------
# 2. Dataset Class
# ------------------------------
class LungDataset(Dataset):
    def __init__(self, root_dir, transform=None, selected_indices=None):
        self.base = datasets.ImageFolder(root=root_dir)
        self.samples = self.base.samples
        self.classes = self.base.classes
        self.class_to_idx = self.base.class_to_idx
        self.transform = transform
        if selected_indices:
            self.samples = [self.samples[i] for i in selected_indices]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# ------------------------------
# 3. Transforms
# ------------------------------
train_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.3, hue=0.02),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ------------------------------
# 4. Paths
# ------------------------------
DATA_ROOT = "DataSet"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR  = os.path.join(DATA_ROOT, "valid")
TEST_DIR = os.path.join(DATA_ROOT, "test")

# ------------------------------
# 5. Base Datasets
# ------------------------------
train_base = LungDataset(TRAIN_DIR)
val_base   = LungDataset(VAL_DIR)
test_base  = LungDataset(TEST_DIR)

NUM_CLASSES = len(train_base.classes)
print(f"✅ Found {NUM_CLASSES} classes: {train_base.classes}")


# ------------------------------
# 6. Data Augmentation + Save New Images
# ------------------------------
from collections import defaultdict
from torchvision import transforms as T
from PIL import Image
import shutil

class_to_idxs = defaultdict(list)
for i, (path, lbl) in enumerate(train_base.samples):
    class_to_idxs[lbl].append((i, path))

AUGMENT_DIR = os.path.join(DATA_ROOT, "augmented")
os.makedirs(AUGMENT_DIR, exist_ok=True)

image_id = 0
aug_train_idx = []

augmentations = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.3),
])

# Save augmented images only for classes with < TARGET_PER_CLASS
for lbl, idx_paths in class_to_idxs.items():
    cls_name = train_base.classes[lbl]
    save_dir = os.path.join(AUGMENT_DIR, cls_name)
    os.makedirs(save_dir, exist_ok=True)

    original_count = len(idx_paths)
    required = TARGET_PER_CLASS - original_count
    sampled = random.choices(idx_paths, k=required) if required > 0 else []

    for _, path in sampled:
        img = Image.open(path).convert("RGB")
        img_aug = augmentations(img)
        save_path = os.path.join(save_dir, f"aug_{image_id}.jpg")
        img_aug.save(save_path)
        image_id += 1

    for i, _ in idx_paths:
        aug_train_idx.append(i)  # original indices retained

# Custom dataset that includes original + augmented
class AugmentedLungDataset(Dataset):
    def __init__(self, original_dataset, aug_dir, transform=None, selected_indices=None):
        self.samples = original_dataset.samples.copy()
        self.transform = transform
        self.class_to_idx = original_dataset.class_to_idx

        for root, _, files in os.walk(aug_dir):
            cls_name = os.path.basename(root)
            if cls_name in self.class_to_idx:
                lbl = self.class_to_idx[cls_name]
                for file in files:
                    if file.endswith(".jpg"):
                        self.samples.append((os.path.join(root, file), lbl))

        if selected_indices:
            self.samples = [self.samples[i] for i in selected_indices]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# Final dataset and loader
train_ds = AugmentedLungDataset(train_base, AUGMENT_DIR, train_transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f"📦 Final Train Dataset (with Augmentation): {len(train_ds)} samples")


val_ds   = LungDataset(VAL_DIR,   val_transform)
test_ds  = LungDataset(TEST_DIR,  val_transform)

val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"📦 Dataset sizes - Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

# --------------------------------------------------
# 1. Transformer building blocks (same as your ViT)
# --------------------------------------------------

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim), nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

class Attention(nn.Module):
    def __init__(self, dim, heads=8, dim_head=64, dropout=0.):
        super().__init__()
        inner_dim = dim_head * heads
        self.heads = heads
        self.scale = dim_head ** -0.5
        self.norm = nn.LayerNorm(dim)
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.attend = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(dropout)
        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x = self.norm(x)
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=self.heads), qkv)
        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        attn = self.attend(dots)
        out  = torch.matmul(attn, v)
        out  = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout=0.):
        super().__init__()
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout),
                FeedForward(dim, mlp_dim, dropout=dropout)
            ]))
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x)   + x
        return self.norm(x)


# -----------------------------------------
# 2. ResNet + ViT hybrid Model Definition
# -----------------------------------------

class ResNetViT(nn.Module):
    def __init__(
        self,
        *,
        image_size,        # e.g. 224
        num_classes,       # from your dataset
        resnet_out_dim=2048,
        vit_dim=512,
        vit_depth=6,
        vit_heads=8,
        vit_mlp_dim=1024,
        vit_dim_head=64,
        dropout=0.1,
        emb_dropout=0.1,
    ):
        super().__init__()
        # 1) ResNet50 backbone up through last conv
        backbone = resnet50(pretrained=True)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])

        # 2) 1×1 conv → ViT embedding dim
        self.proj = nn.Conv2d(resnet_out_dim, vit_dim, kernel_size=1)

        # 3) CLS token + positional embeddings
        # ResNet50 downsamples by 32 → feature map is (image_size/32)² patches
        num_patches = (image_size // 32) ** 2
        self.cls_token     = nn.Parameter(torch.randn(1, 1, vit_dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, vit_dim))
        self.dropout       = nn.Dropout(emb_dropout)

        # 4) Transformer encoder
        self.transformer = Transformer(
            dim=vit_dim,
            depth=vit_depth,
            heads=vit_heads,
            dim_head=vit_dim_head,
            mlp_dim=vit_mlp_dim,
            dropout=dropout
        )

        # 5) Classification head
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(vit_dim),
            nn.Linear(vit_dim, 512), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, img):
        # A) conv features
        x = self.backbone(img)         # (B, 2048, H', W'), H' = W' = image_size/32
        x = self.proj(x)               # (B, vit_dim, H', W')

        # B) flatten to sequence
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)  # (B, H*W, C)

        # C) prepend CLS + add pos embedding
        cls_tokens = repeat(self.cls_token, '1 1 d -> b 1 d', b=B)
        x = torch.cat((cls_tokens, x), dim=1)      # (B, 1+H*W, C)
        x = x + self.pos_embedding[:, : x.size(1)]
        x = self.dropout(x)

        # D) transformer + pool
        x = self.transformer(x)                  # (B, 1+H*W, C)
        x = x.mean(dim=1)                        # mean over tokens

        # E) head
        return self.mlp_head(x)


# -----------------------------------------
# 3. Instantiate and move to device
# -----------------------------------------

IMAGE_SIZE   = 224
NUM_CLASSES  = len(train_base.classes)  # from your earlier Dataset
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ResNetViT(
    image_size=IMAGE_SIZE,
    num_classes=NUM_CLASSES,
    vit_dim=512, vit_depth=6, vit_heads=8,
    vit_mlp_dim=1024, vit_dim_head=64,
    dropout=0.1, emb_dropout=0.1
).to(DEVICE)

print(f"✅ ResNet + ViT hybrid ready → {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")


In [ ]:
from collections import Counter
label_counts = Counter([label for _, label in train_ds])
print(label_counts)  # Should print {0: 1000, 1: 1000} if binary classes

In [ ]:
import os
import torch
from tqdm import tqdm

# ------------------------------
# 0. Checkpoint Config
# ------------------------------
CHECKPOINT_PATH = "resnet_vt_checkpoint.pth"
RESUME          = True           # set False to start fresh even if a ckpt exists

# ------------------------------
# 1. Loss, Optimizer, Scheduler
# ------------------------------
criterion  = nn.CrossEntropyLoss()
optimizer  = optim.AdamW(model.parameters(), lr=LR)
scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# ------------------------------
# 2. Optionally Resume
# ------------------------------
start_epoch   = 0
best_val_acc  = 0.0

if RESUME and os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optim_state"])
    scheduler.load_state_dict(ckpt["sched_state"])
    best_val_acc = ckpt["best_val_acc"]
    start_epoch  = ckpt["epoch"] + 1
    print(f"✅ Resumed from epoch {ckpt['epoch']} | best_val_acc={best_val_acc:.4f}")
else:
    print("ℹ️  No checkpoint found or RESUME=False — starting fresh.")

# ------------------------------
# 3. Train + Validate + Save
# ------------------------------
for epoch in range(start_epoch, NUM_EPOCHS):
    print(f"\n🔄 Epoch {epoch+1}/{NUM_EPOCHS}")
    model.train()
    running_loss = correct_preds = total_preds = 0

    for images, labels in tqdm(train_loader, desc="  • Training"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct_preds += (outputs.argmax(1) == labels).sum().item()
        total_preds   += labels.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    train_acc  = correct_preds / total_preds
    print(f"    🟢 Train  | loss={train_loss:.4f}  acc={train_acc:.4f}")

    # ---------- Validation ----------
    model.eval()
    val_loss = val_correct = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="  • Validate", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_loss /= len(val_loader.dataset)
    val_acc   = val_correct / len(val_loader.dataset)
    print(f"    🔵 Val    | loss={val_loss:.4f}  acc={val_acc:.4f}")

    # ---------- Checkpoint ----------
    improved = val_acc > best_val_acc
    if improved:
        best_val_acc = val_acc
        torch.save(
            {
                "epoch":        epoch,
                "model_state":  model.state_dict(),
                "optim_state":  optimizer.state_dict(),
                "sched_state":  scheduler.state_dict(),
                "best_val_acc": best_val_acc,
            },
            CHECKPOINT_PATH,
        )
        print(f"    💾 Saved new best checkpoint (acc={best_val_acc:.4f})")

    scheduler.step()
    print(f"    🔄 LR stepped -> {scheduler.get_last_lr()[0]:.6f}")

print("\n🎉 Training complete.")


In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE)["model_state"])
model.eval()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

# Run on test data
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_base.classes)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix on Test Set")
plt.show()

# Optional: Classification Report
print("\n🧾 Classification Report:\n", classification_report(all_labels, all_preds, target_names=train_base.classes))


In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries
import numpy as np
from matplotlib import pyplot as plt

# 1. Load one image without transform
raw_test_ds = LungDataset(TEST_DIR, transform=None)
sample_img, sample_label = raw_test_ds[0]
img_np = np.array(sample_img)

# 2. Define predict_proba
def predict_proba(images_np):
    model.eval()
    with torch.no_grad():
        images_resized = torch.nn.functional.interpolate(
            torch.tensor(images_np).permute(0, 3, 1, 2).float(),
            size=(224, 224), mode='bilinear', align_corners=False
        ) / 255.0
        images_resized = T.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225])(images_resized)
        images_resized = images_resized.to(DEVICE)
        outputs = model(images_resized)
        return outputs.softmax(1).cpu().numpy()


# 3. Explain with LIME
explainer = lime_image.LimeImageExplainer()
explanation = explainer.explain_instance(
    image=img_np,
    classifier_fn=predict_proba,
    top_labels=1,
    hide_color=0,
    num_samples=1000
)

# 4. Visualize
temp, mask = explanation.get_image_and_mask(
    label=explanation.top_labels[0],
    positive_only=False,
    hide_rest=False,
    num_features=10,
    min_weight=0.0
)

plt.figure(figsize=(6, 6))
plt.imshow(mark_boundaries(temp / 255.0, mask))
plt.title(f"LIME Explanation for class: {train_base.classes[sample_label]}")
plt.axis('off')
plt.show()
